# Implementing Tools & Tool Calling in LLMs for Agentic AI with LangChain


## Tools for Agentic AI Systems

In Agentic AI systems, tools are functions that allow the large language model (LLM) to do things it cannot do on its own. These include actions like searching for information, looking up data from a database, performing calculations, or calling APIs.

## Tool Calling for Agentic AI Systems

Tool calling allows the LLM to request a tool when it needs help answering a question. In LangChain, this is done using the bind_tools method. This tells the model which tools are available, how they work, and when to use them when input queries are passed to the LLM.

LangGraph can also be leveraged to create reactive agents by passing tools, prompts, and LLMs to built-in functions. This we will be explored in the next notebook or hands-on lab.

Note that the LLM will not call the tools automatically—they must be executed manually or, as demonstrated in the upcoming live demos, by AI Agents.


## Problem Statement

We aim to build **HealthBuddy**, an Agentic AI system designed to assist users with health-related queries through intelligent reasoning and tool use. Leveraging the **ReAct framework** (Reasoning + Acting), HealthBuddy integrates a **Large Language Model (LLM)** with custom tools and structured instructions to provide accurate, actionable, and personalized responses.

HealthBuddy is capable of:

- Answering health-related queries, including symptom checks and treatment suggestions, by using external sources such as **Web Search** and **PubMed Search** tools
- Recommending appropriate doctors for specific medical concerns via a **Doctor Recommendation Tool**
- Displaying available **appointment slots** and **booking appointments** after collecting essential user details (name, email, phone)
- Maintaining **multi-turn conversations** in a natural, chat-like flow
- Supporting **multi-user interactions** with isolated memory contexts for each user to enable personalized and continuous engagement

Each notebook in this series will build and extend different components of the system, progressively enhancing HealthBuddy’s intelligence, tool-use capabilities, and user experience.


### Objective:

The focus of this notebook is to demonstrate how to build custom tools for Agentic AI systems and connect them to a standard Large Language Model (LLM). Specifically, we will implement the following tools:

- **Web Search Tool** for retrieving up-to-date health information 
- **PubMed Search Tool** for accessing scientific medical literature
- **Doctor Recommendation Tool** for suggesting suitable medical professionals based on user symptoms or concerns

These tools will be integrated with the LLM using the ReAct (Reasoning + Acting) approach, enabling the model to invoke tool calls when it requires external information. 

We want to:

1. Understand how to design and register custom tools

2. Enable LLMs to dynamically invoke tools via function/tool calling

This notebook focuses on building tools and demonstrating tool-calling as depicted in the workflow below.


In [ ]:
from IPython.display import Image
Image(filename='/Users/csharm33/code/genai_dbx/Course4/Data/tool_calling.png')


In the next notebook we will reuse this to build our first Agentic AI System.


In [ ]:
#%run /Users/csharm33/code/genai_dbx/Course3/.setup/learner_setup.ipynb

## Load Necessary Dependencies


In [ ]:
import os
from dotenv import load_dotenv
import httpx
import json
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain_core.tools import tool
from IPython.display import display, Markdown
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)


## Setup Authentication and LLM Client

The following code sets up the UAIS environment to authenticate the application and securely connect to the Azure OpenAI services. It retrieves the access token required for authorization and initializes both the LLM and embedding clients using Azure OpenAI authentication.


In [ ]:
def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course4/Data/vars.env')


AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
OPENAI_API_VERSION = os.environ["OPENAI_API_VERSION"]
EMBEDDINGS_DEPLOYMENT_NAME = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]
MODEL_DEPLOYMENT_NAME = os.environ["MODEL_DEPLOYMENT_NAME"]
PROJECT_ID = os.environ['PROJECT_ID']
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")

chat_client = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment=MODEL_DEPLOYMENT_NAME,
    temperature=0,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": PROJECT_ID
    }
)


embeddings_client = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION,
    azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
    azure_ad_token=get_access_token(),
    default_headers={
        "projectId": PROJECT_ID
    }
)

## Prepare Databases for Simulated Web Search and PubMed Search Tools

Now, we will load sample data simulating real web page documents and research papers from PubMed.

Our tools, which we will define later, will reference this data instead of making live internet searches (due to environment and security restrictions preventing internet access).

This simulates real-time web or research paper searches using preloaded documents as the source.

We will create two separate vector databases:

- `web_search_db`: Contains general-purpose health-related web articles, simulating responses to real-time internet queries like “treatments for diabetes” or “how to handle panic attacks”

- `pubmed_db`: Built using sample abstracts and content from medical research papers, simulating PubMed search results for clinically accurate, research-backed information

Later as we proceed in the notebook, agent tools will query these databases to retrieve relevant information and help the LLM provide informed, grounded responses.


In [ ]:
# Loading pre-built datasets for web search and PubMed
with open('/Users/csharm33/code/genai_dbx/Course4/Data/search_data.json', 'r') as f:
    search_docs = json.load(f)

# Display the top-level keys in the loaded dataset
print(f"Major Document Types: {list(search_docs.keys())}")
tiktoken_cache_dir = os.path.abspath("/Users/csharm33/code/genai_dbx/Course4/.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir
os.environ["ANONYMIZED_TELEMETRY"]="False"

# Create a vector database from simulated web search documents.
# Enables semantic search over general health-related content.
web_search_db = Chroma.from_texts(
    search_docs['search_pages'],              # List of web-style document texts
    collection_name='web_search_db_demo1',    # Collection name
    embedding=embeddings_client,              # Embedding model for vectorization
)

# Create a vector database from simulated PubMed documents.
# Enables semantic search over clinical research content.
pubmed_db = Chroma.from_texts(
    search_docs['pubmed_pages'],              # List of PubMed-style document texts
    collection_name='pubmed_db_demo1',        # Collection name
    embedding=embeddings_client,              # Embedding model for vectorization
)


## Preparing Database for Doctor Recommendations Tool

In this section, we will create a small, in-memory database containing information about doctors. This data will be used by our Doctor Recommendation Tool to help users find the right doctor based on their health query or symptoms.

The database includes a list of doctors along with their:

- Name

- Specialization (e.g., Dermatology, Pediatrics, Cardiology)

- Location

- Availability

- Contact information

We will use a simple Python list of dictionaries to store the doctor data. This will keep it easy to understand and modify. In a real-world application, this would typically be replaced by a backend database like PostgreSQL, MongoDB, or an external API.

We will build a tool later to use this data to recommend doctors based on the user’s needs—for example, recommending a pediatrician for a child’s fever or a cardiologist for chest pain.


In [ ]:
# Load doctor dataset with availability, specialization, and contact details
doctors_db = [
    {"name": "Dr. Janet Dyne", "specialization": "Endocrinology (Diabetes Care)", "available_timings": "10:00 AM - 1:00 PM", "location": "City Health Clinic", "contact": "janet.dyne@healthclinic.com"},
    {"name": "Dr. Don Blake", "specialization": "Cardiology (Heart Specialist)", "available_timings": "2:00 PM - 5:00 PM", "location": "Metro Cardiac Center", "contact": "don.blake@metrocardiac.com"},
    {"name": "Dr. Susan D'Souza", "specialization": "Oncology (Cancer Care)", "available_timings": "11:00 AM - 2:00 PM", "location": "Hope Cancer Institute", "contact": "susan.dsouza@hopecancer.org"},
    {"name": "Dr. Matt Murdock", "specialization": "Psychiatry (Mental Health)", "available_timings": "4:00 PM - 7:00 PM", "location": "Mind Care Center", "contact": "matt.murdock@mindcare.com"},
    {"name": "Dr. Dinah Lance", "specialization": "General Physician", "available_timings": "9:00 AM - 12:00 PM", "location": "Downtown Medical Center", "contact": "dinah.lance@downtownmed.com"}
]


## Create Tools for AI Agent

In this section, we will define the tools that our AI Agent will use to perform specific tasks.

LangChain makes it easy to create and register tools using the `Tool` class. The tool includes:
- A name and description
- The python function to be called
- An input schema that tells the model what arguments it can use

When tools are properly defined, they enable the model to solve more complex problems by allowing it to perform actions and access external data. This makes the system more useful and reliable.

### 🧪 Example
```python


In [ ]:
# Tool for simulating a web search on general health topics
@tool
def search_web(query: str) -> list:
    """Search the web for a query. Useful for retrieving general or up-to-date healthcare information."""
    # Perform semantic similarity search over the web search vector database
    results = web_search_db.similarity_search(query, k=5)
    docs = [doc.page_content for doc in results]
    return docs


# Tool for simulating a PubMed-style academic literature search
@tool
def search_pubmed(query: str) -> list:
    """Search PubMed for scientific articles related to the query."""
    # Perform semantic similarity search over the PubMed vector database
    results = pubmed_db.similarity_search(query, k=5)
    docs = [doc.page_content for doc in results]
    return docs


# Tool for recommending a doctor based on user symptoms or health-related queries
@tool
def recommend_doctor(query: str) -> dict:
    """Recommend the most suitable doctor based on the user's symptoms."""
    doctors_list = str(doctors_db)

    # Use the LLM to reason over the list and identify the best match for the user's concern
    prompt = f"""You are an assistant helping recommend a doctor based on a patient's health issues.

Here is the list of available doctors:
{doctors_list}

Given the user's query: "{query}"

Choose the most suitable doctor from the list. Only pick one doctor.
Return only the selected doctor's information in JSON format.
If unsure, recommend the General Physician.
"""

    response = chat_client.invoke(prompt)
    return {"recommended_doctor": response.content}

## Testing out our Tools

You can just call the above tools manually by using the `.invoke(...)` function with the necessary inputs.

The following code shows some example queries and outputs on calling these tools


In [ ]:
results = search_web.invoke('Recent treatments in Diabetes')
print(f"Total documents: {len(results)}")
print()
display(Markdown((results[0][:3000])))
results = search_pubmed.invoke('Recent treatments in Diabetes')
print(f"Total documents: {len(results)}")
print()
display(Markdown((results[1][:3000])))


In [ ]:
result = recommend_doctor.invoke('Treatments for Diabetes')
print(f"Raw Tool Output:\n{json.dumps(result, indent=2)}")
print("-" * 50)
print(f"\nFormatted Tool Output:\n{result['recommended_doctor']}")

## Explore LLM Tool Calling with Custom Tools

An agent is basically an LLM that has the capability to automatically call relevant functions to perform complex or tool-based tasks based on inputs or prompts provided by users.

Tool calling, also popularly known as function calling, is the ability to reliably enable such LLMs to call external tools and APIs.

We will leverage the custom tools created earlier in the previous section to test whether the LLM can automatically call the right tools based on input prompts.


## Tool calling for LLMs


Tool calling allows a model to respond to a given prompt by generating output that matches a user-defined schema. While the name implies that the model is performing some action, that is actually not the case! The model is coming up with the arguments to a tool, and running the tool (or not) is up to the user or the agent as defined by the user.

Many LLM providers, such as Anthropic, Cohere, Google, Mistral, or OpenAI support variants of a tool calling feature. These features typically allow requests to the LLM to include available tools and their schemas, and for responses to include calls to these tools.

For example, if a user asks a question that requires a search or data lookup, the model can decide to call a specific tool with a tool call request.

Tool calling in LangChain works by:
1. Registering defined tool functions using `@tool` decorator
2. Binding the tools to the model using `llm.bind_tools([tool1, tool2, ...])`
3. Passing a user query to the bound model
4. Letting the model decide whether to use a tool or respond directly

This setup lets the model behave more like an agent that can take actions, observe results, and continue the conversation intelligently.

By using `bind_tools`, we give the LLM the ability to understand what tools are available and make requests to use them only when needed.

### 🧪 Example
```python


### Tool Call Requests

LLMs will not call and execute the tools for you directly, but will request tool calls based on reasoning on the input (user query) if they feel that they do not have enough information to answer the question directly.

To experiment with this, let's first define our tools and bind them to our LLM
# List of all tools that the LLM should be aware of


In [ ]:
# These tools were defined earlier using the @tool decorator
tools = [search_web, search_pubmed, recommend_doctor]

# Bind the tools to the LLM so it can invoke them when necessary
# Enables tool-calling functionality in LangChain
llm_with_tools = chat_client.bind_tools(tools=tools)


Now we can test out these prompts using our LLM with tools


In [ ]:
prompts = [
    "treatments available for diabetes",
    "Research papers on diabetes treatments",
    "What doctor could I visit for diabetes",
    "Explain what is diabetes in simple terms"
]

results = []

for prompt in prompts:
    result = llm_with_tools.invoke(prompt)
    results.append(result)


**Note that:**

- If the LLM can answer the question directly, the response will be present in `result.content`
- If the LLM can't answer the question directly, the response will be empty, but will contain one or more tool call requests in `result.tool_calls`


In [ ]:
for prompt, result in zip(prompts, results):
    # If the model provided a direct response without using any tools
    if result.content:
        print('No tool call was needed')
        print(f'Prompt: {prompt}')
        print(f'Direct LLM Response: {result.content}')

    # If the model determined that a tool should be called
    if result.tool_calls:
        print('LLM decided to call tools')
        print(f'Prompt: {prompt}')
        print(f'Tool Call Request: {result.tool_calls}')

    print('-'*50)
    print()


### Tool Call Responses with Manual Tool Calling

While Agentic Frameworks like LangChain or LangGraph can leverage AI agents to handle tool calls by automatically executing them and retrieving results (to feed back to the LLM for further reasoning), here we will manually invoke these tool calls to demonstrate how results appear.

In future demos, AI agents will handle this automatically.

We start by creating a mapping between tool call names (strings) and the actual tool functions (Python function names).


In [ ]:
# Create a mapping between tool call names and their corresponding functions
tool_mapper = {
    "search_web": search_web,
    "search_pubmed": search_pubmed,
    "recommend_doctor": recommend_doctor
}


If you inspect the tool call requests for any input prompt, we have the tool name and the input arguments. These, will need to be extracted and then we will need to call the actual tool with the input arguments

For example based on the following tool call request, we would need to call:

`search_web.invoke(query='treatments available for diabetes')`
# Inspect the tool call request made
results[0].tool_calls


We can easily automate this by extracting the tool call request `name` and `args` and invoking the actual python tool function matching the `name` with the input arguments (`args`)


In [ ]:
for result in results:
    tool_call_requests = result.tool_calls
    for tool_call in tool_call_requests:
        # Retrieve the actual tool function based on the tool name from the LLM
        selected_tool = tool_mapper[tool_call["name"]]

        # Invoke the tool using the arguments provided by the LLM
        print(f"Calling tool: {tool_call['name']}")
        tool_output = selected_tool.invoke(tool_call["args"])
        print(f"Tool Output:\n{json.dumps(tool_output, indent=2)}")
        print("-"*50)
        print()


In the upcoming hands-on demo you will see how to connect LLMs, tools and instruction prompts together to automate this and build your first Agentic AI System with Tool-Use!


## Homework: Build a Healthcare Tool for LLM Tool Calling
**Goal:** Apply your understanding of tool design and registration by creating a simple, custom healthcare tool and enabling an LLM to use it through tool calling.


#### Tasks
1. **Create a Custom Healthcare Tool:**  Build a simple function-based tool relevant to healthcare. Examples include:
    - A BMI calculator
    - A medication-to-condition lookup
    - A basic symptom checker

2. **Register the Tool with the LLM:** Use the @tool decorator and register the tool using bind_tools() so the LLM knows how and when to invoke it.

3. **Test Tool Calling:**  Prompt the LLM with a question that requires using your tool.  Verify that:
    - The LLM chooses to invoke your tool
    - The final response includes the tool output appropriately


#### Example
- **If you create a BMI calculator, test it with a prompt like:**
    - “Can you check if someone weighing 80 kg and 1.75 meters tall is overweight?”

- **Expected behavior:** The LLM should call your tool and return something like:
    - “The BMI is 26.1, which falls in the overweight range.”
